# 01 · 전처리 V2 (기존 피처 + 참고 피처)

> 목표: 기존 02_Preprocess의 모든 피처를 유지하고, **박천상님 참고 노트북의 좋은 파생변수**를 추가한다.
> 단 참고 노트북의 누수(split별 max/mean/scaler)는 고치고, inf/NaN이 안 나게 안전 분모 + clip을 쓴다.
> 결과는 **processed_v2/** 에 저장한다 (기존 processed/ 는 건드리지 않음).

순서: 로드 → 정제 → **분할** → 결측대치(train중앙값) → 파생변수(기존+참고 피처) → 극단값 clip(train기준) → log1p → 표준화(train fit) → 저장
(파생변수를 분할·대치 뒤에 계산해서, lead_time과 train 통계를 누수 없이 쓴다.)

## 0. 환경 설정

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

from utils import set_seed
set_seed(42)

PROJECT_ROOT = Path(r"C:\Users\Administrator\Desktop\딥러닝응용\TermProject")
DATA_DIR  = PROJECT_ROOT / "dataset_1"
OUT_DIR   = PROJECT_ROOT / "processed_v2"
OUT_DIR.mkdir(exist_ok=True)
TRAIN_CSV = DATA_DIR / "Kaggle_Training_Dataset_v2.csv"
TEST_CSV  = DATA_DIR / "Kaggle_Test_Dataset_v2.csv"
TARGET = "went_on_backorder"
print("데이터 존재:", TRAIN_CSV.exists(), TEST_CSV.exists())

데이터 존재: True True


## 1. 원본 로드 + 정제 (행 단위)
기존 02와 동일: 빈 꼬리행 제거, Yes/No->1/0, perf의 -99->결측, 결측 플래그 생성, sku/rev_stop/stop_auto_buy 제거.

In [2]:
def load_csv(path):
    df = pd.read_csv(path, low_memory=False)
    if df.iloc[-1].isnull().sum() > len(df.columns) // 2:
        df = df.iloc[:-1].copy()
    return df

BINARY_COLS = ['potential_issue','deck_risk','oe_constraint','ppap_risk','stop_auto_buy','rev_stop','went_on_backorder']
DROP_COLS   = ['sku','rev_stop','stop_auto_buy']

def clean_rowwise(df):
    df = df.copy()
    for c in BINARY_COLS:
        if c in df.columns:
            df[c] = (df[c] == 'Yes').astype(int)
    for c in ['perf_6_month_avg','perf_12_month_avg']:
        df[c] = df[c].replace(-99, np.nan)
    df['perf_missing'] = df['perf_6_month_avg'].isnull().astype(int)
    df['lead_time_missing'] = df['lead_time'].isnull().astype(int)
    return df.drop(columns=[c for c in DROP_COLS if c in df.columns])

df_train_full = clean_rowwise(load_csv(TRAIN_CSV))
df_test       = clean_rowwise(load_csv(TEST_CSV))
print("정제 후 컬럼:", df_train_full.shape[1])

정제 후 컬럼: 22


## 2. 분할 (train / val / test)
파생변수·통계 계산 전에 먼저 나눈다. 이래야 train 통계만으로 안전하게 처리할 수 있다.
백오더가 0.67%뿐이라 stratify로 비율 유지.

In [3]:
train_df, val_df = train_test_split(df_train_full, test_size=0.2, random_state=42,
                                    stratify=df_train_full[TARGET])
test_df = df_test.copy()
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name:5s}: {len(d):>9,}  양성 {d[TARGET].mean():.3%}")

train: 1,350,288  양성 0.669%
val  :   337,572  양성 0.669%
test :   242,075  양성 1.110%


## 3. 결측치 대치 (train 중앙값으로만)
lead_time, perf의 결측을 train 중앙값으로 채운다. 중앙값은 train에서만 계산해 val/test에 적용(누수 방지).

In [4]:
num_cols = train_df.select_dtypes(include=[np.number]).columns
num_cols = [c for c in num_cols if c != TARGET]
medians = train_df[num_cols].median()

for d in (train_df, val_df, test_df):
    d[num_cols] = d[num_cols].fillna(medians)
print("남은 결측:", int(train_df.isnull().sum().sum()), int(val_df.isnull().sum().sum()), int(test_df.isnull().sum().sum()))

남은 결측: 0 0 0


## 4. 기존 파생변수 (02와 동일, 12개)
원진 팀장님 제안 5개를 포함한 기존 파생변수. (분할·대치 뒤지만 이 변수들은 lead_time을 안 써서 결과 동일)

In [5]:
def add_base_features(df):
    df = df.copy()
    df['available_inventory'] = df['national_inv'] - df['min_bank']
    df['real_available_inventory'] = df['national_inv'] - df['min_bank'] - df['local_bo_qty']
    df['future_available_inventory_3m'] = (df['national_inv'] + df['in_transit_qty']
                                           - df['min_bank'] - df['forecast_3_month'])
    df['shortage_ratio_3m'] = ((df['forecast_3_month'] - df['national_inv'] - df['in_transit_qty'])
                               / (df['forecast_3_month'] + 1))
    df['no_sales_but_demand_signal'] = (((df['sales_6_month'] == 0)
                                         & ((df['forecast_3_month'] > 0) | (df['local_bo_qty'] > 0))).astype(int))
    df['neg_inv_flag'] = (df['national_inv'] < 0).astype(int)
    df['has_local_bo'] = (df['local_bo_qty'] > 0).astype(int)
    df['has_past_due'] = (df['pieces_past_due'] > 0).astype(int)
    df['below_safety_flag'] = (df['national_inv'] < df['min_bank']).astype(int)
    df['sales_acceleration'] = np.where(df['sales_3_month'] > 0,
                                        (df['sales_1_month'] * 3 - df['sales_3_month']) / df['sales_3_month'], 0)
    df['forecast_accuracy'] = np.where(df['forecast_3_month'] > 0,
                                       df['sales_3_month'] / df['forecast_3_month'], 0)
    df['total_risk_score'] = (df['neg_inv_flag'] * 3 + df['has_local_bo'] * 2
                              + df['potential_issue'] * 2 + df['has_past_due'] + df['below_safety_flag'])
    return df

train_df = add_base_features(train_df)
val_df   = add_base_features(val_df)
test_df  = add_base_features(test_df)
print("기존 파생 후 컬럼:", train_df.shape[1])

기존 파생 후 컬럼: 34


## 5. 박천상님 참고 피처 (누수 고침 + 안전 분모)
박천상님 참고 공식을 가져오되:
- 분모가 0/음수가 안 되게 안전 분모 사용(national_inv는 clip(lower=0)+1 등).
- product_criticality의 sales_importance는 참고 노트북처럼 max로 나누는데, **그 max를 train에서만** 계산(누수 방지).

In [6]:
F = ['forecast_3_month','forecast_6_month','forecast_9_month']

def add_chunsang_reference_features(df, avg_sales_max):
    df = df.copy()
    safe_inv = df['national_inv'].clip(lower=0) + 1            # 분모용 (>=1)
    df['inventory_coverage']    = df['national_inv'] / (df['forecast_3_month'] + 1)
    df['stock_shortage_risk']   = (df['forecast_3_month'] - df['in_transit_qty']) / safe_inv
    df['days_of_stock']         = df['national_inv'] / (df['sales_1_month'] / 30 + 0.1)
    df['supply_chain_health']   = df['in_transit_qty'] / (safe_inv + df['in_transit_qty'])
    df['lead_time_demand_ratio'] = df['lead_time'] * df['forecast_3_month'] / 90
    df['replenishment_urgency'] = df['forecast_3_month'] / (safe_inv + df['in_transit_qty'])
    df['avg_sales'] = (df['sales_1_month'] * 0.4 + df['sales_3_month'] * 0.3
                       + df['sales_6_month'] * 0.2 + df['sales_9_month'] * 0.1)
    df['forecast_volatility'] = df[F].std(axis=1) / (df[F].mean(axis=1) + 1)
    sales_importance = df['avg_sales'] / avg_sales_max         # max는 train에서만
    df['product_criticality'] = sales_importance * df['lead_time'] * df['forecast_volatility']
    return df

# avg_sales의 max를 train에서만 계산
avg_sales_train = (train_df['sales_1_month'] * 0.4 + train_df['sales_3_month'] * 0.3
                   + train_df['sales_6_month'] * 0.2 + train_df['sales_9_month'] * 0.1)
AVG_SALES_MAX = float(avg_sales_train.max())
print("train avg_sales max:", round(AVG_SALES_MAX, 2))

train_df = add_chunsang_reference_features(train_df, AVG_SALES_MAX)
val_df   = add_chunsang_reference_features(val_df, AVG_SALES_MAX)
test_df  = add_chunsang_reference_features(test_df, AVG_SALES_MAX)
print("참고 피처 파생 후 컬럼:", train_df.shape[1])

train avg_sales max: 1223675.9


참고 피처 파생 후 컬럼: 43


## 6. 극단값 clip (train 분위수 기준)
참고 피처 비율 변수는 분모가 작을 때 큰 값이 나올 수 있다. train의 0.5% ~ 99.5% 범위로 잘라(clip)
극단값을 눌러준다. clip 경계는 **train에서만** 계산해 val/test에 적용.

In [7]:
REFERENCE_COLS = ['inventory_coverage','stock_shortage_risk','days_of_stock','supply_chain_health',
           'lead_time_demand_ratio','replenishment_urgency','avg_sales','forecast_volatility','product_criticality']

low = train_df[REFERENCE_COLS].quantile(0.005)
high = train_df[REFERENCE_COLS].quantile(0.995)
for d in (train_df, val_df, test_df):
    d[REFERENCE_COLS] = d[REFERENCE_COLS].clip(lower=low, upper=high, axis=1)

# inf/NaN 최종 점검
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    bad = (~np.isfinite(d[REFERENCE_COLS].to_numpy())).sum()
    print(f"{name}: 참고피처 비정상값 {bad}개")

train: 참고피처 비정상값 0개
val: 참고피처 비정상값 0개
test: 참고피처 비정상값 0개


## 7. log1p (수량형) + 표준화 (train fit)
- log1p: 한쪽으로 길게 쏠린 수량 변수만(기존 02와 동일 목록 + avg_sales).
- 표준화: 모든 피처를 평균0/표준편차1. fit은 train만, val/test엔 적용만.

In [8]:
LOG_COLS = ['national_inv','in_transit_qty','forecast_3_month','forecast_6_month','forecast_9_month',
            'sales_1_month','sales_3_month','sales_6_month','sales_9_month','min_bank','pieces_past_due',
            'local_bo_qty','available_inventory','real_available_inventory','future_available_inventory_3m','avg_sales']

def apply_log1p(df):
    df = df.copy()
    for c in LOG_COLS:
        df[c] = np.sign(df[c]) * np.log1p(np.abs(df[c]))
    return df

train_df = apply_log1p(train_df); val_df = apply_log1p(val_df); test_df = apply_log1p(test_df)

feature_cols = [c for c in train_df.columns if c != TARGET]
scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
val_df[feature_cols]   = scaler.transform(val_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])
print("표준화 완료. 총 피처:", len(feature_cols))

표준화 완료. 총 피처: 42


## 8. 저장 (processed_v2/) + 검증
processed_v2/에 저장. 기존 processed/ 와 별개. 재로드해서 inf/NaN 없고 비율 유지되는지 확인.

In [9]:
train_df.to_pickle(OUT_DIR / "train.pkl")
val_df.to_pickle(OUT_DIR / "val.pkl")
test_df.to_pickle(OUT_DIR / "test.pkl")
joblib.dump({'scaler': scaler, 'medians': medians, 'feature_cols': feature_cols,
             'avg_sales_max': AVG_SALES_MAX, 'clip_low': low, 'clip_high': high},
            OUT_DIR / "preprocess_artifacts.joblib")

import numpy as np
tr = pd.read_pickle(OUT_DIR / "train.pkl")
print("저장 파일:", [p.name for p in sorted(OUT_DIR.glob('*'))])
print("피처 수:", len(feature_cols), "(기존 02는 33개 -> 참고 피처 9개 추가)")
print("inf/NaN 없음:", bool(np.isfinite(tr[feature_cols].to_numpy()).all()))
print("양성 비율 train/val/test: {:.3%} / {:.3%} / {:.3%}".format(
      tr[TARGET].mean(), val_df[TARGET].mean(), test_df[TARGET].mean()))

저장 파일: ['preprocess_artifacts.joblib', 'test.pkl', 'train.pkl', 'val.pkl']
피처 수: 42 (기존 02는 33개 -> 참고 피처 9개 추가)


inf/NaN 없음: True
양성 비율 train/val/test: 0.669% / 0.669% / 1.110%


---
### 다음 단계
processed_v2/ 생성 완료. 다음은 02_Baseline_Tree_V2(XGBoost/LGBM 튜닝)로 v2 피처 효과를 본다.
모든 v2 모델은 이 processed_v2 데이터를 쓰고 results_v2.csv에 기록한다.